In [ ]:
import os
import matplotlib.pyplot as plt
import cv2

# 1. Numărăm câte imagini avem
train_path = 'datasets/train/images'
valid_path = 'datasets/valid/images'

num_train = len(os.listdir(train_path))
num_valid = len(os.listdir(valid_path))

print(f"📊 Statistica Setului de Date:")
print(f" - Imagini de antrenament: {num_train}")
print(f" - Imagini de validare: {num_valid}")
print(f" - Total: {num_train + num_valid}")

# 2. Creăm un grafic simplu (Pie Chart) pentru documentație
labels = ['Antrenament', 'Validare']
sizes = [num_train, num_valid]
colors = ['#66b3ff','#99ff99']

plt.figure(figsize=(5, 5))
plt.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
plt.title('Distribuția Datelor')
plt.show()

# 3. Afișăm o imagine de exemplu
sample_img_name = os.listdir(train_path)[0]
sample_img_path = os.path.join(train_path, sample_img_name)

img = cv2.imread(sample_img_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(8, 6))
plt.imshow(img)
plt.title(f"Exemplu Imagine: {sample_img_name}")
plt.axis('off')
plt.show()

In [ ]:
from ultralytics import YOLO

# 1. Încărcăm modelul Nano (cel mai rapid)
model = YOLO('yolov8n.pt')

# 2. Antrenăm modelul
# epochs=25 este un echilibru bun între viteză și precizie pentru proiect
results = model.train(
    data='datasets/data_video.yaml',
    epochs=25,
    imgsz=640,
    plots=True
)

print("✅ Antrenament finalizat!")

In [ ]:
from IPython.display import Image, display
import os

# YOLO salvează automat graficele în folderul 'runs'
# Căutăm cel mai recent folder de antrenament
runs_dir = 'runs/detect'
# Sortăm folderele ca să îl găsim pe ultimul creat (train, train2, train3...)
dirs = [d for d in os.listdir(runs_dir) if 'train' in d]
latest_run = sorted(dirs, key=lambda x: os.path.getmtime(os.path.join(runs_dir, x)))[-1]

results_img_path = os.path.join(runs_dir, latest_run, 'results.png')
confusion_matrix_path = os.path.join(runs_dir, latest_run, 'confusion_matrix.png')

print(f"Afișăm rezultatele din: {latest_run}")

# Afișăm graficele de performanță
if os.path.exists(results_img_path):
    display(Image(filename=results_img_path))
else:
    print("Graficul results.png nu a fost găsit.")

# Afișăm matricea de confuzie (cât de des confundă clasele)
if os.path.exists(confusion_matrix_path):
    display(Image(filename=confusion_matrix_path))
else:
    print("Matricea de confuzie nu a fost găsită.")

In [ ]:
from ultralytics import YOLO
import os
import cv2
import matplotlib.pyplot as plt
import random

# 1. Încărcăm modelul tău antrenat
model_path = 'runs/detect/train/weights/best.pt' # Verifică dacă folderul e train, train2 etc.

if os.path.exists(model_path):
    model = YOLO(model_path)
    print(f"✅ Model încărcat: {model_path}")
else:
    print("❌ Nu găsesc modelul! Verifică calea.")
    model = None

# 2. Alegem folderul cu poze de test (sau validare)
# Poți schimba 'valid' cu 'test' dacă ai folderul test populat
images_folder = 'datasets/valid/images'

if model and os.path.exists(images_folder):
    # Luăm toate pozele din folder
    all_images = [f for f in os.listdir(images_folder) if f.endswith(('.jpg', '.png', '.jpeg'))]

    # Alegem 5 poze la întâmplare
    selected_images = random.sample(all_images, min(5, len(all_images)))

    print(f"🔍 Testăm pe {len(selected_images)} imagini aleatoare...")

    plt.figure(figsize=(15, 10))

    for i, img_name in enumerate(selected_images):
        img_path = os.path.join(images_folder, img_name)

        # Facem predicția
        results = model(img_path)

        # Desenăm rezultatul pe imagine (plot() returnează imaginea cu pătrate)
        res_plotted = results[0].plot()

        # Convertim din BGR (OpenCV) în RGB (Matplotlib) ca să se vadă culorile corect
        res_rgb = cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB)

        # Afișăm
        plt.subplot(1, 5, i+1) # Facem un rând cu 5 poze
        plt.imshow(res_rgb)
        plt.axis('off')
        plt.title(img_name, fontsize=8)

    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Ceva nu e în regulă cu folderul de imagini sau modelul.")

In [3]:
from ultralytics import YOLO

# 1. Încărcăm "CREIERUL" TĂU deja antrenat
# Verifică dacă calea e corectă (train, train2 etc.)
path_to_old_model = 'runs/detect/train/weights/best.pt'

print(f"🧠 Încărcăm modelul vechi din: {path_to_old_model}")
model = YOLO(path_to_old_model)

# 2. Antrenăm DOAR pe datele noi
# Modelul va păstra ce știe, dar se va adapta la noile poze
results = model.train(
    data='datasets_video/data_video.yaml',  # <--- Specificăm folderul unde se află
    epochs=30,
    imgsz=640,
    plots=True,
    batch=16,
    lr0=0.01,
    name='train_finetune'
)    # Se va salva într-un folder nou


print("✅ Fine-tuning terminat! Noul model e în 'runs/detect/train_finetune/weights/best.pt'")

🧠 Încărcăm modelul vechi din: runs/detect/train/weights/best.pt
Ultralytics 8.4.14  Python-3.13.7 torch-2.10.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets_video/data_video.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs/detect/train/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train_finetune6, nbs=64,